# 01 – Xử lý dữ liệu GEFCom2012 (Giai đoạn 2)

**Checklist:** 2.1 → 2.13, 2.16 (bản sửa theo nhận xét của thầy ngày 01/11)

**Mục tiêu.** Từ các file CSV gốc của track Load, tạo ra một bảng theo giờ cho từng chuỗi cần dự báo (chuỗi tổng hệ thống hoặc từng zone), gồm: phụ tải, nhiệt độ, biến lịch.

**Những điểm thầy nhận xét cần thể hiện trong notebook này:**
1. Chuỗi tổng phải **tự cộng từ 20 zone**, và phải **kiểm tra thiếu ở từng zone trước khi cộng** (một zone thiếu là tổng sai).
2. 11 trạm nhiệt độ không gắn sẵn với zone nào → việc chọn trạm là **một quyết định thiết kế** và phải ghi lý do.
3. Không chỉ làm chuỗi tổng: `load_dataset()` phải trả được bảng cho **từng zone**.

**Đầu vào:** `data/raw/` (copy 6 file trong `GEFCOM2012_Data/Load/`).
**Đầu ra:** các file trong `data/processed/`. Toàn bộ code xử lý nằm ở `src/data.py`; notebook này gọi từng bước và kiểm tra kết quả.

In [1]:
import sys
sys.path.append("..")          # để import được thư mục src/ khi notebook nằm trong notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (11, 3.5)

from src.data import *

## 1. Đọc dữ liệu phụ tải thô

`Load_history.csv` ghi số kiểu `"16,853"` (dấu phẩy ngăn cách hàng nghìn), nên phải đọc với `thousands=","`, nếu không pandas sẽ hiểu cả cột là chuỗi ký tự. Mỗi dòng là **1 zone × 1 ngày**, 24 cột `h1..h24` là 24 giờ.

In [2]:
hist, sol = read_load_raw()
print("Load_history :", hist.shape, "| số zone:", hist.zone_id.nunique())
print("Load_solution:", sol.shape, "| zone:", sorted(sol.zone_id.unique()))
hist.head(3)

Load_history : (33000, 28) | số zone: 20
Load_solution: (1323, 28) | zone: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21)]


,zone_id,year,month,day,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24
0,1,2004,1,1,16853.0,16450.0,16517.0,16873.0,17064.0,17727.0,18574.0,19355.0,19534.0,18611.0,17666.0,16374.0,15106.0,14455.0,13518.0,13138.0,14130.0,16809.0,18150.0,18235.0,17925.0,16904.0,16162.0,14750.0
1,1,2004,1,2,14155.0,14038.0,14019.0,14489.0,14920.0,16072.0,17800.0,19089.0,19577.0,20047.0,19770.0,18564.0,18137.0,17046.0,16127.0,15448.0,15839.0,17727.0,18895.0,18650.0,18443.0,17580.0,16467.0,15258.0
2,1,2004,1,3,14439.0,14272.0,14109.0,14081.0,14775.0,15491.0,16536.0,18197.0,19109.0,18012.0,17200.0,15950.0,14978.0,14162.0,13507.0,13414.0,13826.0,15825.0,16996.0,16394.0,15406.0,14278.0,13315.0,12424.0


## 2. (2.13) Báo cáo ô thiếu theo từng zone – TRƯỚC khi cộng chuỗi tổng

Đếm số giờ bị thiếu của từng zone trong `Load_history`. Nếu các zone thiếu ở những thời điểm khác nhau thì không thể cộng thẳng 20 zone được.

In [3]:
rep = missing_report(hist)
rep.to_csv("../results/tables/2_13_thieu_theo_zone.csv")
print("Số giờ thiếu khác nhau giữa các zone:", rep["n_missing_hours"].unique())
rep

Số giờ thiếu khác nhau giữa các zone: [1530]


,n_missing_hours,first_missing,last_missing
zone_id,,,
1,1530,2005-03-06,2008-07-07 23:00:00
2,1530,2005-03-06,2008-07-07 23:00:00
3,1530,2005-03-06,2008-07-07 23:00:00
4,1530,2005-03-06,2008-07-07 23:00:00
5,1530,2005-03-06,2008-07-07 23:00:00
6,1530,2005-03-06,2008-07-07 23:00:00
7,1530,2005-03-06,2008-07-07 23:00:00
8,1530,2005-03-06,2008-07-07 23:00:00
9,1530,2005-03-06,2008-07-07 23:00:00


In [4]:
# Các giờ thiếu nằm ở đâu? Gom các ngày thiếu liên tiếp thành từng đoạn
long1 = wide_to_long(hist[hist.zone_id == 1], "zone_id")
miss_days = long1.loc[long1["value"].isna(), "datetime"].dt.normalize().drop_duplicates().sort_values()
blocks = (miss_days.diff() != pd.Timedelta(days=1)).cumsum()
doan = miss_days.groupby(blocks).agg(["min", "max", "count"]).rename(columns={"count": "so_ngay"})
doan

,min,max,so_ngay
datetime,,,
1,2005-03-06,2005-03-12,7
2,2005-06-20,2005-06-26,7
3,2005-09-10,2005-09-16,7
4,2005-12-25,2005-12-31,7
5,2006-02-13,2006-02-19,7
6,2006-05-25,2006-05-31,7
7,2006-08-02,2006-08-08,7
8,2006-11-22,2006-11-28,7
9,2008-06-30,2008-07-07,8


**Nhận xét.**
- Cả 20 zone thiếu **đúng cùng các giờ**: 9 đoạn 7 ngày (8 tuần ban tổ chức xóa để thi backcast, cộng 1 tuần cuối 01–07/07/2008 để thi dự báo) và một phần ngày 30/06/2008.
- 9 tuần có đáp án trong `Load_solution.csv` → lấp lại được.
- Riêng **18 giờ ngày 30/06/2008 (06:00–23:00)** không có trong cả hai file → phải tự lấp.
- Vì các zone thiếu đồng thời, sau khi lấp xong mới cộng chuỗi tổng thì tổng không bị sai lệch.

## 3. Ghép solution, chuyển sang bảng dài, lấp 18 giờ còn thiếu

- **Bảng dài:** mỗi dòng là 1 giờ của 1 zone. Quy ước nhãn đầu khoảng: `h1` = khoảng 00:00–01:00 → gán thời điểm 00:00.
- **18 giờ còn thiếu:** lấp bằng giá trị **cùng giờ tuần trước** (lag 168), vì phụ tải lặp theo tuần. Các giờ này được gắn cờ `is_imputed = True` và **sẽ không được dùng để chấm điểm**. Thực tế ngày 30/06/2008 cũng nằm sau ngày cuối của tập test (29/06/2008), nên không ảnh hưởng kết quả.

In [5]:
zones = build_zones_long()
print(zones.shape, "= 20 zone ×", zones.shape[0] // 20, "giờ")
print("giờ lấy từ solution:", zones["filled_from_solution"].sum() // 20, "giờ / zone")
print("giờ lấp bằng lag 168:", zones["is_imputed"].sum() // 20, "giờ / zone")
zones.head()

(792000, 5) = 20 zone × 39600 giờ
giờ lấy từ solution: 1512 giờ / zone
giờ lấp bằng lag 168: 18 giờ / zone


,zone_id,datetime,load,filled_from_solution,is_imputed
0,1,2004-01-01 00:00:00,16853.0,False,False
1,1,2004-01-01 01:00:00,16450.0,False,False
2,1,2004-01-01 02:00:00,16517.0,False,False
3,1,2004-01-01 03:00:00,16873.0,False,False
4,1,2004-01-01 04:00:00,17064.0,False,False


## 4. Kiểm tra chuỗi tổng

Trong `Load_solution`, zone 21 là tổng hệ thống do ban tổ chức tính. Nếu tổng tự cộng của mình khớp zone 21 ở mọi giờ có đáp án thì cách ghép là đúng.

In [6]:
print("Chênh lệch lớn nhất |tổng 20 zone − zone 21| =", check_system_vs_zone21(zones))

Chênh lệch lớn nhất |tổng 20 zone − zone 21| = 0.0


## 5. Nhiệt độ 11 trạm

`temperature_history.csv` dừng ở 30/06/2008 05:00. Phần còn lại (06:00 ngày 30/06 → hết 07/07) có trong `temperature_solution.csv`, nên ghép vào để chuỗi nhiệt độ liên tục. Đơn vị là °F.

In [7]:
temp = build_temperature()
print(temp.shape, "| ô thiếu:", temp.isna().sum().sum())
st = [f"T_s{i}" for i in range(1, 12)]
c = temp[st].corr().values
print(f"Tương quan giữa các trạm: thấp nhất {c[np.triu_indices(11, 1)].min():.3f}, cao nhất {c[np.triu_indices(11, 1)].max():.3f}")
temp.describe().T.round(1)

(39600, 12) | ô thiếu: 0
Tương quan giữa các trạm: thấp nhất 0.941, cao nhất 0.998


,count,mean,std,min,25%,50%,75%,max
T_s1,39600.0,59.5,16.7,12.0,46.0,61.0,73.0,103.0
T_s2,39600.0,54.6,17.4,0.0,41.0,56.0,68.0,95.0
T_s3,39600.0,55.9,17.6,8.0,42.0,57.0,70.0,98.0
T_s4,39600.0,60.0,17.4,11.0,46.0,62.0,73.0,103.0
T_s5,39600.0,56.5,17.6,6.0,43.0,57.0,71.0,99.0
T_s6,39600.0,58.6,17.1,7.0,45.0,60.0,72.0,103.0
T_s7,39600.0,58.7,18.0,8.0,45.0,60.0,73.0,104.0
T_s8,39600.0,59.1,17.8,6.0,45.0,61.0,73.0,103.0
T_s9,39600.0,56.5,17.9,5.0,42.0,57.0,71.0,100.0
T_s10,39600.0,58.6,17.4,5.0,45.0,60.0,72.0,102.0


## 6. Ngày lễ

Mỗi ô trong `Holiday_List.csv` có dạng `"Monday, January 16"`, năm nằm ở tên cột. Riêng Tết 2005 được nghỉ bù ngày `"Friday, December 31, 2004"` nên ô đó có ghi năm. Kiểm tra lại: thứ ghi trong file phải trùng với thứ tính từ ngày.

In [8]:
hol = read_holidays()
print("Số ngày lễ:", len(hol), "| tỉ lệ khớp thứ:", hol["weekday_ok"].mean())
hol.head(10)

Số ngày lễ: 45 | tỉ lệ khớp thứ: 1.0


,holiday,date,weekday_in_file,weekday_ok
0,New Year's Day,2004-01-01,Thursday,True
1,New Year's Day,2004-12-31,Friday,True
2,New Year's Day,2006-01-02,Monday,True
3,New Year's Day,2007-01-01,Monday,True
4,New Year's Day,2008-01-01,Tuesday,True
5,"Birthday of Martin Luther King, Jr.",2004-01-19,Monday,True
6,"Birthday of Martin Luther King, Jr.",2005-01-17,Monday,True
7,"Birthday of Martin Luther King, Jr.",2006-01-16,Monday,True
8,"Birthday of Martin Luther King, Jr.",2007-01-15,Monday,True
9,"Birthday of Martin Luther King, Jr.",2008-01-21,Monday,True


## 7. Chọn nhiệt độ cho chuỗi tổng (2.11) – một quyết định thiết kế

Cách chọn: với mỗi trạm (và trung bình 11 trạm `T_mean`), hồi quy `load ~ T + T² + T³` và so sánh R². **Chỉ dùng dữ liệu đến `SELECTION_END` (30/06/2006)** để việc chọn không nhìn thấy validation/test.

In [9]:
system = zones.groupby("datetime")["load"].sum()
r2 = station_table(system, temp)
r2.to_frame("R2").round(3).T

,T_mean,T_s3,T_s5,T_s7,T_s9,T_s2,T_s11,T_s6,T_s10,T_s1,T_s4,T_s8
R2,0.629,0.621,0.606,0.606,0.601,0.592,0.584,0.582,0.579,0.571,0.551,0.538


**Quyết định:** dùng `T_mean` cho chuỗi tổng, vì R² cao nhất. Điều này hợp lý về mặt vật lý: chuỗi tổng là tổng của cả vùng nên nhiệt độ trung bình toàn vùng đại diện tốt hơn một trạm đơn lẻ. Với từng zone, cùng tiêu chí này được dùng để chọn trạm riêng (lưu trong `zone_station.json`, xem notebook 02).

## 8. Lưu tất cả và kiểm tra `load_dataset()` (2.16)

In [10]:
mapping = build_all()
df = load_dataset("system")
print("\nsystem:", df.shape, df.index.min(), "→", df.index.max())
d10 = load_dataset(10)
print("zone 10:", d10.shape, "| nhiệt độ dùng:", mapping["10"])
df.head()

zones_long : (792000, 5)
temperature: (39600, 12) | thiếu: 0
holidays   : 45 ngày | thứ khớp: 1.0
chênh lệch tổng 20 zone vs zone 21: 0.0

system: (39600, 20) 2004-01-01 00:00:00 → 2008-07-07 23:00:00
zone 10: (39600, 20) | nhiệt độ dùng: T_s3


,load,is_imputed,T_s1,T_s2,T_s3,T_s4,T_s5,T_s6,T_s7,T_s8,T_s9,T_s10,T_s11,T_mean,hour,dayofweek,month,dayofyear,is_holiday,T
datetime,,,,,,,,,,,,,,,,,,,,
2004-01-01 00:00:00,1397668.0,False,46.0,38.0,44.0,45.0,42.0,44.0,45.0,43.0,41.0,42.0,36.0,42.363636,0,3,1,1,1,42.363636
2004-01-01 01:00:00,1351507.0,False,46.0,36.0,42.0,43.0,42.0,43.0,44.0,44.0,39.0,43.0,32.0,41.272727,1,3,1,1,1,41.272727
2004-01-01 02:00:00,1315742.0,False,45.0,35.0,40.0,41.0,40.0,42.0,41.0,42.0,36.0,43.0,31.0,39.636364,2,3,1,1,1,39.636364
2004-01-01 03:00:00,1308023.0,False,41.0,30.0,36.0,37.0,39.0,38.0,40.0,34.0,35.0,39.0,30.0,36.272727,3,3,1,1,1,36.272727
2004-01-01 04:00:00,1344117.0,False,39.0,30.0,34.0,33.0,40.0,38.0,35.0,30.0,33.0,35.0,34.0,34.636364,4,3,1,1,1,34.636364


## Tổng kết

| Kiểm tra | Kết quả |
|---|---|
| Thiếu theo zone | mọi zone thiếu cùng 1 530 giờ, lấp được hết |
| Tổng 20 zone so với zone 21 | khớp tuyệt đối |
| Nhiệt độ | 39 600 giờ, không thiếu |
| Ngày lễ | 45 ngày, khớp thứ 100% |

`load_dataset(series)` trả về cùng một cấu trúc cho chuỗi tổng (`"system"`) và từng zone (1..20). Cột `T` là nhiệt độ đã chọn cho chuỗi đó. Các notebook sau chỉ dùng hàm này.